# Project 07 -- Stress Testing Framework Walkthrough

This notebook walks through the end-to-end stress testing framework:

1. **Synthetic data generation** -- macro factors and portfolio returns
2. **Transmission model** -- OLS fit and sensitivity analysis
3. **DFAST regulatory scenarios** -- baseline / adverse / severely adverse
4. **Historical crisis replay** -- GFC 2008, COVID 2020, SVB 2023
5. **Reverse stress test** -- find the break-point scenario
6. **Key visualizations** -- scenario paths, loss waterfall, capital impact

In [ ]:
import sys
from pathlib import Path

# Add project src/ to path
_SRC = str(Path.cwd().parent / "src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from diagnostics import (
    plot_capital_impact,
    plot_historical_comparison,
    plot_loss_waterfall,
    plot_scenario_paths,
)
from model import StressTestFramework
from reverse_stress import sensitivity_analysis
from scenarios import get_dfast_scenarios, get_historical_scenarios

# Framework imports
from transmission import (
    MacroTransmissionModel,
)

print("All imports OK")

## 1. Generate Synthetic Macro Data and Portfolio Returns

We simulate 40 quarters (10 years) of six macro factors and construct
portfolio returns as a linear combination plus noise. The true betas
are known, so we can verify the transmission model recovers them.

In [ ]:
rng = np.random.default_rng(42)
n = 40

macro_factors = pd.DataFrame({
    "gdp_growth": rng.normal(0.02, 0.01, n),
    "unemployment": rng.normal(0.05, 0.01, n),
    "equity_index": rng.normal(0.02, 0.05, n),
    "interest_rate_10y": rng.normal(0.03, 0.005, n),
    "credit_spread": rng.normal(0.02, 0.005, n),
    "house_price_index": rng.normal(0.01, 0.02, n),
})

# True linear relationship: return = factors @ betas + noise
betas_true = np.array([0.5, -0.3, 0.8, -0.2, -0.6, 0.3])
portfolio_returns = (macro_factors.values @ betas_true) + rng.normal(0, 0.01, n)

print(f"Macro factors shape: {macro_factors.shape}")
print(f"Portfolio returns shape: {portfolio_returns.shape}")
print(f"\nTrue betas: {dict(zip(macro_factors.columns, betas_true))}")
macro_factors.describe().round(4)

## 2. Fit Transmission Model and Show Sensitivity Table

The `MacroTransmissionModel` runs OLS regression of portfolio returns on macro
factors. With synthetic data the estimated betas should closely match the true
values and R-squared should be high.

In [ ]:
model = MacroTransmissionModel()
model.fit(portfolio_returns, macro_factors)

print(f"R-squared:    {model.r_squared:.4f}")
print(f"Residual std: {model.residual_std:.6f}")
print("\nEstimated betas vs true betas:")
for name, est, true in zip(model.factor_names, model.betas, betas_true):
    print(f"  {name:20s}  estimated={est:+.4f}  true={true:+.4f}")

print("\n--- Sensitivity Table ---")
sens_table = model.sensitivity_table()
sens_table

## 3. Run DFAST Stress Test and Show Results

The DFAST module provides three regulatory-style scenarios (baseline, adverse,
severely adverse). For each, we compute cumulative loss, maximum quarterly loss,
and post-stress capital ratio impact.

In [ ]:
# Set up framework with config
config = {
    "transmission": {
        "factors": list(macro_factors.columns),
    },
    "capital": {"initial_ratio": 0.12},
    "credit": {"lgd": 0.45, "stress_factor": 2.0},
    "reverse_stress": {"loss_threshold": 0.15},
}

fw = StressTestFramework(config)
dfast_results = fw.run_dfast(portfolio_returns, macro_factors)

print("=== DFAST Scenario Results ===\n")
print(dfast_results.to_string(index=False))
print("\nLoss ordering check:")
losses = dfast_results.set_index("scenario")["cumulative_loss"]
print(f"  Baseline:         {losses['baseline']:.4f}")
print(f"  Adverse:          {losses['adverse']:.4f}")
print(f"  Severely Adverse: {losses['severely_adverse']:.4f}")
dfast_results

## 4. Run Historical Scenarios and Compare

We replay three historical crises -- GFC 2008, COVID 2020, and SVB 2023 --
through the fitted transmission model. Each scenario specifies single-period
factor shocks calibrated to observed peak moves.

In [ ]:
hist_results = fw.run_historical(portfolio_returns, macro_factors)

print("=== Historical Scenario Results ===\n")
for _, row in hist_results.iterrows():
    print(f"  {row['scenario']:12s}  predicted loss = {row['predicted_loss']:+.4f}")

print("\n--- Raw shocks used ---")
historical_shocks = get_historical_scenarios()
for name, shocks in historical_shocks.items():
    print(f"\n  {name}:")
    for factor, val in shocks.items():
        print(f"    {factor:20s} = {val:+.4f}")

hist_results

## 5. Reverse Stress Test

The reverse stress test finds the *minimum-norm* macro shock vector that causes
portfolio loss to exceed a given threshold (here 10%). This answers the
question: "What is the smallest plausible scenario that breaks us?"

In [ ]:
threshold = 0.10
reverse_result = fw.run_reverse(portfolio_returns, macro_factors, threshold=threshold)

print(f"=== Reverse Stress Test (threshold = {threshold:.0%}) ===\n")
print(f"  Success:        {reverse_result['success']}")
print(f"  Predicted loss: {reverse_result['predicted_loss']:.4f}")
print(f"  Shock norm:     {reverse_result['shock_norm']:.4f}")
print("\n  Optimal shocks:")
for factor, shock in reverse_result["optimal_shocks"].items():
    print(f"    {factor:20s} = {shock:+.4f}")

# Also run sensitivity analysis around the optimal shocks
sens_around_optimal = sensitivity_analysis(
    model, reverse_result["optimal_shocks"], list(macro_factors.columns)
)
print("\n--- Sensitivity around optimal reverse-stress shocks ---")
sens_around_optimal

## 6. Key Visualizations

Four diagnostic plots produced by the `diagnostics` module:
1. **Scenario paths** -- DFAST factor trajectories over 9 quarters
2. **Loss waterfall** -- factor-by-factor loss decomposition
3. **Historical comparison** -- bar chart of crisis losses
4. **Capital impact** -- post-stress CET1 ratios vs regulatory minimum

In [ ]:
# 6a. Scenario paths -- GDP growth across DFAST scenarios
dfast_scenarios = get_dfast_scenarios()
fig, ax = plot_scenario_paths(dfast_scenarios, "gdp_growth")
plt.show()

In [ ]:
# 6b. Loss waterfall -- factor contributions via sensitivity analysis
base_shocks = {f: 0.0 for f in macro_factors.columns}
sens_df = sensitivity_analysis(model, base_shocks, list(macro_factors.columns))
fig, ax = plot_loss_waterfall(sens_df)
plt.show()

In [ ]:
# 6c. Historical comparison bar chart
fig, ax = plot_historical_comparison(hist_results)
plt.show()

In [ ]:
# 6d. Capital impact by DFAST scenario
fig, ax = plot_capital_impact(dfast_results)
plt.show()

In [ ]:
# Generate full markdown report
report = fw.generate_report()
print(report)